# Lab 02 — Tiền xử lý văn bản tiếng Việt
**Môn Xử lý ngôn ngữ tự nhiên | Nhóm Ba Sáu**

Pipeline: chuẩn hóa Unicode → chữ thường → làm sạch ký tự → **tách từ** → loại stopword → gắn nhãn từ loại → chuẩn hóa từ.
Mỗi bước có giải thích ngắn. Điểm khác tiếng Anh cần nhớ: tiếng Việt viết cách nhau bằng khoảng trắng giữa **âm tiết**, không phải giữa **từ** ("sinh viên", "đại học" là một từ gồm hai âm tiết), nên bước tách từ phải dùng công cụ riêng cho tiếng Việt.

**Mục lục:** Bước 0 Cài đặt · Bước 1 Chuẩn hóa và làm sạch · Bước 2 Tách từ · Bước 3 Stopword · Bước 4 Gắn nhãn từ loại · Bước 5 Chuẩn hóa từ (lemmatization) · Bước 6 Tổng kết

## Bước 0 — Cài đặt

In [ ]:
# ── Cell 1: Cài đặt thư viện (Colab) ──────────────────────────────────────
!pip install underthesea

In [2]:
# ── Cell 2: Import và câu mẫu ─────────────────────────────────────────────
import re
import unicodedata
from underthesea import word_tokenize, pos_tag   # công cụ tách từ và gắn nhãn từ loại cho tiếng Việt

text_vi = "Sinh viên!!! Đang học NLP tại Trường Đại Học Sư Phạm TP.HCM"
text_vi

d:\Data Analysis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'Sinh viên!!! Đang học NLP tại Trường Đại Học Sư Phạm TP.HCM'

## Bước 1 — Chuẩn hóa và làm sạch

**NFC:** chữ có dấu có thể được lưu bằng một mã Unicode (NFC) hoặc bằng chữ cái + dấu rời (NFD). Hai dạng nhìn giống hệt nhưng so sánh chuỗi cho kết quả khác, nên phải đưa về một dạng trước khi xử lý.

In [3]:
# ── Cell 3: Chuẩn hóa Unicode (NFC) và chữ thường ─────────────────────────
text_norm = unicodedata.normalize('NFC', text_vi)
text_lower = text_norm.lower()
text_lower   # kỳ vọng: 'sinh viên!!! đang học nlp tại trường đại học sư phạm tp.hcm' 

'sinh viên!!! đang học nlp tại trường đại học sư phạm tp.hcm'

In [4]:
# ── Cell 4: Minh hoạ NFC và NFD ───────────────────────────────────────────
nfd = unicodedata.normalize('NFD', 'học')
nfc = unicodedata.normalize('NFC', 'học')
print(nfd == nfc, '| độ dài NFD:', len(nfd), '| độ dài NFC:', len(nfc))   # False, 4 và 3

False | độ dài NFD: 4 | độ dài NFC: 3


**Làm sạch ký tự:** giữ chữ cái a–z, chữ có dấu tiếng Việt và khoảng trắng; mọi ký tự khác (dấu câu, số, ký hiệu) thay bằng khoảng trắng rồi gộp khoảng trắng thừa.
Lưu ý: bước này cũng làm mất số và dấu gạch; với bài toán cần giữ số (giá, năm) thì phải sửa lại biểu thức.

In [5]:
# ── Cell 5: Làm sạch bằng regex ───────────────────────────────────────────
VI_CHARS = ('a-z'
            'àáảãạăắằẳẵặâầấẩẫậ'
            'èéẻẽẹêềếểễệ'
            'ìíỉĩị'
            'òóỏõọôồốổỗộơờớởỡợ'
            'ùúủũụưừứửữự'
            'ỳýỷỹỵđ')
text_clean = re.sub(rf'[^{VI_CHARS}\s]', ' ', text_lower)
text_clean = re.sub(r'\s+', ' ', text_clean).strip()
text_clean   # kỳ vọng: 'sinh viên đang học nlp tại trường đại học sư phạm tp hcm' 

'sinh viên đang học nlp tại trường đại học sư phạm tp hcm'

## Bước 2 — Tách từ (word segmentation)

Tách theo khoảng trắng chỉ cho **âm tiết**: `sinh`, `viên`, `đại`, `học`, ... Công cụ tách từ tiếng Việt (`underthesea`) ghép các âm tiết thành từ: `sinh viên`, `đại học`, `sư phạm`.
(Trước đây notebook dùng `nltk.word_tokenize`, vốn thiết kế cho tiếng Anh, nên chỉ ra âm tiết.)

In [6]:
# ── Cell 6: So sánh tách âm tiết và tách từ ───────────────────────────────
syllables = text_clean.split()     # tương đương cách tách của word_tokenize trên văn bản đã làm sạch
tokens = word_tokenize(text_clean)
print('Âm tiết (', len(syllables), '):', syllables)
print('Từ      (', len(tokens), '):', tokens)   # kỳ vọng có 'sinh viên', 'đại học', 'sư phạm' 

Âm tiết ( 13 ): ['sinh', 'viên', 'đang', 'học', 'nlp', 'tại', 'trường', 'đại', 'học', 'sư', 'phạm', 'tp', 'hcm']
Từ      ( 10 ): ['sinh viên', 'đang', 'học', 'nlp', 'tại', 'trường', 'đại học', 'sư phạm', 'tp', 'hcm']


## Bước 3 — Loại stopword

Stopword là những từ rất phổ biến, ít mang nội dung (đang, tại, là, của, và...). Danh sách dưới đây chỉ là **danh sách nhỏ để minh hoạ**; bài toán thật dùng danh sách stopword tiếng Việt đầy đủ hơn.
Dùng `set` thay vì `list` để kiểm tra thuộc về nhanh hơn (O(1) so với O(n)).
Stopword được viết theo đơn vị **từ**, nên phải loại sau khi tách từ.

In [7]:
# ── Cell 7: Loại stopword ─────────────────────────────────────────────────
stop_words_vi = {"tại", "đang", "là", "của", "và", "các", "những", "một", "cho", "với"}
tokens_nostop = [w for w in tokens if w not in stop_words_vi]
print('Bị loại :', [w for w in tokens if w in stop_words_vi])
print('Còn lại :', tokens_nostop)

Bị loại : ['đang', 'tại']
Còn lại : ['sinh viên', 'học', 'nlp', 'trường', 'đại học', 'sư phạm', 'tp', 'hcm']


## Bước 4 — Gắn nhãn từ loại (POS tagging)

`nltk.pos_tag` là bộ gắn nhãn **tiếng Anh**: dùng cho tiếng Việt cho nhãn vô nghĩa (ví dụ gắn "nlp" là tính từ). Phải dùng bộ gắn nhãn tiếng Việt (`underthesea.pos_tag`).
Nhãn thường gặp: N danh từ, V động từ, A tính từ, R phó từ, E giới từ, Np danh từ riêng. Bộ gắn nhãn không hoàn hảo, đặc biệt với từ viết tắt hoặc từ mượn như "nlp".

In [8]:
# ── Cell 8: Gắn nhãn từ loại tiếng Việt ───────────────────────────────────
# Gắn nhãn trên câu đã làm sạch (trước khi loại stopword, vì ngữ cảnh đầy đủ giúp nhãn chính xác hơn)
pos_tag(text_clean)

[('sinh viên', 'N'),
 ('đang', 'R'),
 ('học', 'V'),
 ('nlp', 'V'),
 ('tại', 'E'),
 ('trường', 'N'),
 ('đại học', 'N'),
 ('sư phạm', 'N'),
 ('tp', 'N'),
 ('hcm', 'M')]

## Bước 5 — Chuẩn hóa từ (stemming/lemmatization)

Tiếng Anh biến hình (plays, played, playing → play) nên cần stemming hoặc lemmatization. **Tiếng Việt không biến hình**: "học" luôn là "học" dù ở thì nào, nên bước này **không cần** và lemma của một từ chính là nó.
Ở bước này ta chỉ ghi rõ điều đó, không biến đổi gì thêm.

In [9]:
# ── Cell 9: Lemma tiếng Việt (giữ nguyên) ─────────────────────────────────
lemmas = tokens_nostop   # tiếng Việt không biến hình: lemma = chính từ đó
lemmas

['sinh viên', 'học', 'nlp', 'trường', 'đại học', 'sư phạm', 'tp', 'hcm']

## Bước 6 — Tổng kết

| Bước | Kết quả |
|---|---|
| Chuẩn hóa + lowercase | `sinh viên!!! đang học nlp tại trường đại học sư phạm tp.hcm` |
| Làm sạch | `sinh viên đang học nlp tại trường đại học sư phạm tp hcm` |
| Tách từ | `['sinh viên', 'đang', 'học', 'nlp', 'tại', 'trường', 'đại học', 'sư phạm', 'tp', 'hcm']` |
| Loại stopword | bỏ `đang`, `tại` |

**Kiến thức cần nhớ:**
- NFC gộp chữ + dấu thành một mã Unicode; văn bản chép từ nhiều nguồn có thể lẫn NFC/NFD nên phải chuẩn hóa trước khi so khớp.
- Pipeline phụ thuộc ngôn ngữ: tiếng Anh cần lemmatization, tiếng Việt cần tách từ.
- Dùng `set` khi kiểm tra stopword.